---
# Chapter 6 — The Memory Nexus

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 6: The Memory Nexus |
| Central question | Once an AI has several legitimate ways to access memory, what decides which memory process runs? |
| Main concepts | Memory Nexus, Capability routing, Routing regret |
| Implementation | memory_nexus |
| Experiment | ch6-20260919-nexus (E6 suite) |
| Evidence status | Developmental: optimisation / control |
| Depends on | Chapter 5 (pathways), Chapter 3 (baseline) |

---


## What this notebook demonstrates

The Nexus routes the same query to multiple memory capabilities and exercises the real routing policy. The notebook:

1. **Runs the offline control demo** (`memory_nexus.demo`): rule routing, sequential control, stopping, traces — mechanism only, no model server
2. **Aggregates the frozen E6 matrix** by capability: measured quality against cost
3. **Shows the E6-B headroom result**: quality headroom 0.0 — routing does not beat the best fixed mechanism on quality

> **Evidence status**: Developmental. Routing earned a governance/control role, not a quality role.


## The chapter question

> **Who chooses how we remember?**

Once retrieval, graph search, association and raw-evidence fallback all exist, each query faces a choice. The Nexus makes that choice explicit, auditable, and cost-aware — or the choice stays implicit inside a fixed pipeline nobody named.


## Concepts in this chapter


In [ ]:
import sys
from pathlib import Path


def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent


REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(6)
concepts = (meta.get("chapter", {}).get("concepts")
            or meta.get("concepts", []))
render_table([
    {"Concept ID": c["id"], "Name": c["name"], "Status": c["status"]}
    for c in concepts
], "Chapter 6 Concepts")

## The running example

The same canonical queries meet different capabilities: a decision lookup suits RAG, an influence question suits graph-local traversal, a theme question suits global synthesis, an unfinished-work question suits association. No single capability owns every family — the frozen matrix (E6-A) shows unique wins spread across five of them.


## The mechanism: capabilities, policies, control

A **capability registry** names what can run (RAG, graph-local/global, associative, raw evidence, none) with cost tiers and health. A **policy** (fixed, rules, sequential, classifier, oracle) chooses. A **controller** executes, observes, stops, and records the trace. Routing state carries only query-derived signals — never evaluator fields.


In [ ]:
from memory_nexus import demo as D

print("Stub registry capabilities:", sorted(D.COSTS.keys()))
print("Stub costs (units):", D.COSTS)
print("Demo queries:", [q for _, q in D.QUERIES])
print()
D.run_demo()

In [ ]:
import json
from collections import defaultdict

suite_path = (REPO_ROOT / "experiments" / "benchmark" / "runs"
              / "ch6-20260919-nexus" / "e6-suite.json")
suite = json.loads(suite_path.read_text(encoding="utf-8"))
cells_m = suite["matrix"]["cells"]
print(f"E6 suite {suite['suite_version']}: {len(cells_m)} measured cells")
print("Limitations:", suite["limitations"])

agg = defaultdict(lambda: {"n": 0, "cost": 0.0, "lat": 0.0, "recall": []})
for c in cells_m:
    a = agg[c["capability_id"]]
    a["n"] += 1
    a["cost"] += c.get("cost_units", 0.0)
    a["lat"] += c.get("latency_ms", 0.0)
    if "source_recall" in c.get("metrics", {}):
        a["recall"].append(c["metrics"]["source_recall"])
render_table(
    [{"Capability": k, "Cells": v["n"],
      "Mean cost": round(v["cost"] / v["n"], 2),
      "Mean latency ms": round(v["lat"] / v["n"]),
      "Mean source recall": round(sum(v["recall"]) / len(v["recall"]), 3)}
     for k, v in sorted(agg.items())],
    "Frozen E6 matrix by capability")

dom = suite["experiments"]["E6-A"]["dominance"]
print("\nE6-A unique wins:", dom["unique_wins"])
head = suite["experiments"]["E6-B"]["headroom"]
print(f"E6-B: best fixed={head['best_fixed_by_quality']} "
      f"quality_headroom={head['quality_headroom']} "
      f"(oracle utility {head['oracle_utility']} vs "
      f"best-fixed {head['best_fixed_utility']})")

## What happened?

No single capability dominates: E6-A spreads unique wins across ASSOCIATIVE (4), GRAPH_LOCAL, NONE, RAG and RAW_EVIDENCE (1 each). But E6-B finds **quality headroom 0.0** — the oracle ties the best fixed mechanism on quality. Routing therefore earns a governance role (explicit, auditable, cost-aware choice) rather than a quality role. A fixed policy the designer never named is still a routing decision.


## Connect this to the experiment

The frozen `e6-suite.json` carries the registry manifest, routing families, per-cell measurements and the E6-A/B/CDE analyses. The classifier strand (E6-CDE) trained on 7 held-in samples with a 1-task transfer test — reported as a limitation, not a result. DRIFT is unmeasured in the matrix.


## What this establishes

- **Routing is control**: explicit choice with recorded reason, cost and trace
- **No quality win over the best fixed mechanism** (headroom 0.0)
- **Capability specialisation is real but partial** (spread unique wins)
- **State carries no evaluator fields**; cost tiers are explicit


## What this does NOT establish

- A learned quality router (the classifier strand is 7 samples + 1 transfer task)
- DRIFT routing (unmeasured)
- Real-corpus routing quality


In [ ]:
# TRY IT YOURSELF: route a new query with the rule policy against a
# fixed-RAG floor. Same state builder the experiments use.
from memory_nexus.capabilities.registry import RAG
from memory_nexus.demo import build_stub_registry, QUERIES
from memory_nexus.policies.fixed import FixedPolicy
from memory_nexus.policies.rules import RulePolicy
from memory_nexus.state.features import build_state

registry = build_stub_registry()
query = "What decided the event-store backend and why?"
state = build_state("try-1", query, tuple(registry.available_ids()))
for name, policy in (("rules", RulePolicy()),
                     ("fixed-RAG", FixedPolicy(RAG))):
    action = policy.decide(state, registry)
    print(f"{name:10s} -> {action.capability} ({action.retrieval_budget}): {action.reason}")

## Where this leads next

Chapter 7 asks what licenses a remembered claim: retrieval causality is not evidential support, and no router choice can convert one into the other.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory-chapter.ipynb)
